In [1]:
import pandas as pd
import os
from openpyxl import load_workbook
from openpyxl.styles import Font
from openpyxl.utils import get_column_letter

# === Load the Excel file ===
file_path = r"C:\Users\MajedAljamrah\Downloads\NY_MEM_DATA_FINAL_FLIPA2026-04-17.xlsx"
df = pd.read_excel(file_path)

# === Report date for output file names ===
report_date = "2026-04-17"

# === Mapping Dictionary ===
practice_mapping = {
    'ANTHONY L JORDAN HEALTH CORP': 'Jordan Health',
    'EAST HILL FAMILY MEDICAL': 'East Hill Family Medical',
    'FAMILY HEALTH NETWORK/CTRL NY': 'Family Health Network',
    'FINGER LAKES MIGRANT HEALTH CARE PROJECT': 'Finger Lakes Community Health',
    'FINGERLAKES MIGRANT HLT CA PR': 'Finger Lakes Community Health',
    'MOSAIC HEALTH': 'Mosaic Health',
    'MOSAIC HEALTH INC': 'Mosaic Health',
    'MOUNT VERNON NEIGHBORHOOD HEALTH CENTER': 'Westchester CHC',
    'MT VERNON NEIGHBORHOOD HLTH CTR': 'Westchester CHC',
    'NORTHERN OSWEGO COUNTY HEALTH SRVS': 'ConnextCare',
    'OAK ORCHARD COMMUNITY HLTH CENTER': 'Oak Orchard Community Health',
    'OPEN DOOR FAMILY MEDICAL CENTER': 'Open Door Family Medical',
    'PULASKI HEALTH CENTER(FQHC)': 'ConnextCare',
    'CONNEXTCARE-MMS- SBHC': 'ConnextCare',
    'ROCHESTER PRIMARY CARE NTWK': 'Mosaic Health',
    'ROCHESTER PRIMARY CARE NTWK/PROF FEES': 'Mosaic Health',
    'SYRACUSE COMMUNITY HEALTH CENTER INC': 'Syracuse Community Health',
    'SYRACUSE COMMUNITY HLTH CTR': 'Syracuse Community Health',
    'UCPA ASSOCIATES/NORTH COUNTRY': 'CHC of the North Country',
    'UPSTATE FAMILY HEALTH CENTER': 'Upstate Family Health',
    'WHITNEY M YOUNG JR HEALTH CENTER': 'Whitney M. Young Health',
    'WHITNEY M YOUNG JR HLTH CTR-PROF FEES': 'Whitney M. Young Health',

    'Northern Oswego County Health Services Inc': 'ConnextCare',
    'Upstate Family Health Center, Inc.': 'Upstate Family Health',
    'Oak Orchard Community Health Center, Inc': 'Oak Orchard Community Health',
    'United Cerebral Palsy Association of the North Country': 'CHC of the North Country',
    'Family Health Network of CNY, Inc.': 'Family Health Network',
    'Anthony L. Jordan Health Center': 'Jordan Health',
    'East Hill Family Medical, Inc': 'East Hill Family Medical',
    'Rochester Primary Care Network Inc': 'Mosaic Health',
    'Open Door Family Medical Center, Inc.': 'Open Door Family Medical',
    'Syracuse Community Health Center': 'Syracuse Community Health',
    'Whitney M. Young Jr. Health Center, Inc.': 'Whitney M. Young Health',
    'Finger Lakes Migrant Health Care Project, Inc.': 'Finger Lakes Community Health',

    'FAMILY HEALTH NETWORK OF CNY': 'Family Health Network',
    'ANTHONY L JORDAN HLTH CTR': 'Jordan Health',
    'ANTHONY L JORDAN HLTH CTR 160977295': 'Jordan Health',
    'EAST HILL FAMILY MEDICAL 160983042': 'East Hill Family Medical',
    'FAMILY HEALTH NETWORK OF CNY 161133983': 'Family Health Network',
    'FINGER LAKES MIGRANT HEALTH CARE PROJECT 161581104': 'Finger Lakes Community Health',
    'FINGER LAKES COMMUNITY HEALTH': 'Finger Lakes Community Health',
    'MOSAIC HEALTH INC 161293681': 'Mosaic Health',
    'MOUNT VERNON NEIGHBORHOOD HEALTH CENTER 133315508': 'Westchester CHC',
    'NORTHERN OSWEGO COUNTY HEALTH SRVS 237036393': 'ConnextCare',
    'OAK ORCHARD COMMUNITY HLTH CENTER 161020913': 'Oak Orchard Community Health',
    'OPEN DOOR FAMILY MEDICAL CENTER 132813103': 'Open Door Family Medical',
    'SYRACUSE COMMUNITY HLTH CTR 161080039': 'Syracuse Community Health',
    'UNITED CEREBRAL PALSY ASC/NORTH COUNTRY 161568985': 'CHC of the North Country',
    'UCPA OF THE NORTH COUNTRY': 'CHC of the North Country',
    'UPSTATE FAMILY HEALTH CENTER 474829539': 'Upstate Family Health',
    'WHITNEY M YOUNG JR HEALTH CENTER INC 132922147': 'Whitney M. Young Health',
    'TRILLIUM HEALTH': 'Trillium Health',
    'HIS BRANCHES': 'His Branches'
}

# === Replace Practice Name with mapped health center name ===
df['Practice Name'] = df['Practice Name'].map(practice_mapping).fillna('Unmapped')

# === Output folder ===
output_folder = r"C:\Users\MajedAljamrah\Downloads\NY_MEM_DATA_FINAL_FLIPA2026-04-17 splitted"
os.makedirs(output_folder, exist_ok=True)

# === Beautify function ===
def beautify_excel(filepath):
    wb = load_workbook(filepath)

    for ws in wb.worksheets:
        # Bold headers
        for cell in ws[1]:
            cell.font = Font(bold=True)

        # Freeze top row
        ws.freeze_panes = "A2"

        # Auto-fit columns
        for col in ws.columns:
            max_len = 0
            col_letter = get_column_letter(col[0].column)

            for cell in col:
                if cell.value is not None:
                    max_len = max(max_len, len(str(cell.value)))

            ws.column_dimensions[col_letter].width = min(max_len + 2, 45)

    wb.save(filepath)

# === Split & Save ===
for name, group in df.groupby('Practice Name'):

    # Clean health center name so it can be used safely as a file name
    safe_name = (
        str(name)
        .replace('/', '_')
        .replace('\\', '_')
        .replace(':', '_')
        .replace('*', '_')
        .replace('?', '_')
        .replace('"', '_')
        .replace('<', '_')
        .replace('>', '_')
        .replace('|', '_')
    )

    # Output file name format:
    # [Health Center Name] Member Data Report [Date].xlsx
    filename = f"{safe_name} Member Data Report {report_date}.xlsx"
    output_path = os.path.join(output_folder, filename)

    # Main full health center data
    full_data = group.copy()

    # New tab 1: HC Engaged
    # Patient Engagement = Established Patient
    # Top 10 patients by Total Expense
    hc_engaged = (
        group[group['Patient Engagement'] == 'Established Patient']
        .sort_values(by='Total Expense', ascending=False)
        .head(10)
    )

    # New tab 2: HC Non-Engaged
    # Patient Engagement = No TIN Visit or Other TIN Visit
    # Top 10 patients by Total Expense
    hc_non_engaged = (
        group[group['Patient Engagement'].isin(['No TIN Visit', 'Other TIN Visit'])]
        .sort_values(by='Total Expense', ascending=False)
        .head(10)
    )

    # Save full data and the two new tabs into one Excel file
    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        full_data.to_excel(writer, sheet_name='Full Data', index=False)
        hc_engaged.to_excel(writer, sheet_name='HC Engaged', index=False)
        hc_non_engaged.to_excel(writer, sheet_name='HC Non-Engaged', index=False)

    # Format the Excel file
    beautify_excel(output_path)

print(f"✅ All files saved and formatted to:\n{output_folder}")

✅ All files saved and formatted to:
C:\Users\MajedAljamrah\Downloads\NY_MEM_DATA_FINAL_FLIPA2026-04-17 splitted
